# Predicción del caudal medio diario en cuencas de Colombia con una LSTM regional

**Proyecto final — Análisis con Deep Learning, Universidad de los Andes**
Santiago Palacios Loaiza · Diego Leonardo Barón Espitia · Andrés Felipe Mosquera Hernández · Juan Sebastián Rodríguez Avella

**Parte 2: modelado.** Este notebook parte de la base que genera `proyecto-final-preprocesamiento.ipynb` en `data/processed/`:

1. **Base para el modelo:** lee la base, verifica que esté completa y crea las listas de cuencas.
2. **Configuración del modelo:** define y explica cada hiperparámetro.
3. **Prueba de humo:** entrenamiento corto (20 cuencas, 2 épocas) para verificar que todo funciona.

> **Regla del proyecto:** el modelo **no** recibe caudal observado como entrada (ni `Q(t-1)` ni nada derivado del caudal).
> Es simulación a partir del clima, y eso es lo que permite usarlo en cuencas sin estación de aforo.

## 0. Entorno

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import xarray as xr
import yaml

ROOT = Path.cwd()
DIR_BASE = ROOT / "data" / "processed"   # base generada por proyecto-final-preprocesamiento.ipynb
assert (DIR_BASE / "time_series").exists(), "Ejecuta primero proyecto-final-preprocesamiento.ipynb"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

**Instalación (una sola vez).** PyTorch debe tener soporte CUDA para entrenar en la GPU. Si ya hay un PyTorch solo-CPU instalado, pip no lo reemplaza: hay que forzar la reinstalación (`--force-reinstall`) con el kernel recién reiniciado. La celda siguiente debe mostrar `True` y el nombre de la GPU.

In [ ]:
#import sys
#!{sys.executable} -m pip install neuralhydrology
#!{sys.executable} -m pip install torch==2.13.0 torchvision==0.28.0 --index-url https://download.pytorch.org/whl/cu126 --force-reinstall --no-deps


Looking in indexes: https://download.pytorch.org/whl/cu126



[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.13.0+cu126 True NVIDIA GeForce RTX 2060 with Max-Q Design


## 1. Base para el modelo

El notebook de preprocesamiento deja en `data/processed/`:

| Archivo | Contenido | Uso |
|---|---|---|
| `time_series/<cuenca>.nc` | Forzantes y caudal en mm/día, 1981–2022, una serie por cuenca | NeuralHydrology, al entrenar y evaluar |
| `attributes/atributos_modelo.csv` | Los 48 atributos estáticos | NeuralHydrology |
| `camels_col_diario.parquet` | Tabla única: lo anterior más el período, el estado del caudal y la marca de interpolación | Períodos y cobertura (1.1); análisis de resultados |
| `resumen_cuencas.csv` | Región y días con caudal por período de cada cuenca | Listas de cuencas (1.2) |

Las variables y las fechas de la configuración (sección 2) se toman de estos archivos, no se escriben a mano: el modelo usa exactamente lo que dejó el preprocesamiento. Si este cambia, basta con volver a ejecutar este notebook.

### 1.1 Lectura de la base

Las variables salen de los archivos que lee NeuralHydrology y los períodos, de la tabla. Se verifica que todos los archivos describan las mismas cuencas y variables.

In [3]:
OBJETIVO = "streamflow"   # caudal en mm/día

# Variables del modelo: las que NeuralHydrology encontrará en los archivos
archivos_nc = sorted((DIR_BASE / "time_series").glob("*.nc"))
with xr.open_dataset(archivos_nc[0]) as ds:
    FORZANTES = [v for v in ds.data_vars if v != OBJETIVO]
atributos_modelo = pd.read_csv(DIR_BASE / "attributes" / "atributos_modelo.csv",
                               dtype={"gauge_id": str}, index_col="gauge_id")
ATRIBUTOS = atributos_modelo.columns.tolist()

# Tabla diaria (solo las columnas que se usan aquí) y resumen por cuenca
RUTA_TABLA = DIR_BASE / "camels_col_diario.parquet"
diario = pd.read_parquet(RUTA_TABLA, columns=["gauge_id", "date", "periodo", OBJETIVO])
resumen = pd.read_csv(DIR_BASE / "resumen_cuencas.csv", dtype={"gauge_id": str})
cuencas = resumen["gauge_id"].tolist()

# Períodos: primer y último día de cada etiqueta de la tabla (partición temporal del preprocesamiento, 1.9)
PERIODOS = {p: (diario.loc[diario["periodo"] == p, "date"].min(), diario.loc[diario["periodo"] == p, "date"].max())
            for p in ["train", "validation", "test"]}

# Todos los archivos deben describir las mismas cuencas y variables
assert [a.stem for a in archivos_nc] == sorted(cuencas), "Las series .nc no coinciden con resumen_cuencas.csv"
assert sorted(atributos_modelo.index) == sorted(cuencas), "atributos_modelo.csv no coincide con resumen_cuencas.csv"
assert set(diario["gauge_id"].astype(str)) == set(cuencas), "La tabla diaria no coincide con resumen_cuencas.csv"
assert set(FORZANTES + ATRIBUTOS) <= set(pq.read_schema(RUTA_TABLA).names), "Faltan variables en la tabla diaria"
assert not atributos_modelo.isna().any().any(), "Hay atributos vacíos"

print(f"{len(cuencas)} cuencas | {len(FORZANTES)} forzantes + {len(ATRIBUTOS)} atributos = "
      f"{len(FORZANTES) + len(ATRIBUTOS)} entradas por día")
print("Forzantes:", FORZANTES)

248 cuencas | 4 forzantes + 48 atributos = 52 entradas por día
Forzantes: ['pr', 'poten_evapo', 't_min', 't_max']


In [4]:
# Cobertura de cada período
con_caudal = diario[diario[OBJETIVO].notna()]
pd.DataFrame({
    "inicio": [PERIODOS[p][0].date() for p in PERIODOS],
    "fin": [PERIODOS[p][1].date() for p in PERIODOS],
    "cuencas con caudal": [con_caudal.loc[con_caudal["periodo"] == p, "gauge_id"].nunique() for p in PERIODOS],
    "días con caudal": [int((con_caudal["periodo"] == p).sum()) for p in PERIODOS],
}, index=list(PERIODOS))

,inicio,fin,cuencas con caudal,días con caudal
train,1981-01-01,2010-12-31,248,2267252
validation,2011-01-01,2015-12-31,213,320301
test,2016-01-01,2022-12-31,189,388971


### 1.2 Listas de cuencas

NeuralHydrology recibe las cuencas de cada fase en un archivo de texto, un id por línea:
- `configs/basins/todas.txt`: las 248 cuencas, para el modelo principal.
- `configs/basins/humo_20.txt`: 20 cuencas para la prueba de humo, 4 por región elegidas al azar entre las que tienen caudal en los tres períodos.

In [5]:
DIR_CUENCAS = ROOT / "configs" / "basins"
DIR_CUENCAS.mkdir(parents=True, exist_ok=True)
(DIR_CUENCAS / "todas.txt").write_text("\n".join(resumen["gauge_id"]) + "\n")

con_tres_periodos = resumen[(resumen["dias_q_train"] > 0) & (resumen["dias_q_validation"] > 0) & (resumen["dias_q_test"] > 0)]
humo = con_tres_periodos.groupby("region").sample(n=4, random_state=42).sort_values(["region", "gauge_id"])
(DIR_CUENCAS / "humo_20.txt").write_text("\n".join(humo["gauge_id"]) + "\n")
humo[["gauge_id", "region", "dias_q_train", "dias_q_validation", "dias_q_test"]]

,gauge_id,region,dias_q_train,dias_q_validation,dias_q_test
217,42067010,Amazon,6927,1643,1825
218,42077010,Amazon,7279,1775,1786
219,42077020,Amazon,7539,1816,1818
223,47017100,Amazon,10341,1818,1461
26,15067150,Caribbean,681,1627,2080
27,16017020,Caribbean,10770,1766,2552
30,16027280,Caribbean,5897,1656,1829
40,16057040,Caribbean,10508,1774,731
65,21197010,Magdalena-Cauca,10709,1818,2471
111,23147020,Magdalena-Cauca,10668,1795,2470


## 2. Configuración del modelo

La configuración se define en Python (2.5) y se escribe en `configs/base.yml`, que es lo que lee NeuralHydrology.

### 2.1 Datos y períodos

| Parámetro | Valor | Qué hace |
|---|---|---|
| `dataset`, `data_dir` | `generic`, `data/processed` | Lector genérico de NeuralHydrology, con los archivos de la sección 1 |
| `train/validation/test_basin_file` | `configs/basins/todas.txt` | Las mismas 248 cuencas en las tres fases: la validación es temporal |
| Fechas de inicio y fin | 1981–2010, 2011–2015, 2016–2022 | Partición temporal del preprocesamiento, leída de la tabla (1.1), en formato `DD/MM/YYYY` |
| `dynamic_inputs` | `pr`, `poten_evapo`, `t_min`, `t_max` | Forzantes que el modelo lee cada día de la ventana |
| `static_attributes` | 48 atributos | Se repiten cada día junto a las forzantes: 52 entradas por día |
| `target_variables` | `streamflow` | Caudal en mm/día |
| `clip_targets_to_zero` | `streamflow` | Las predicciones negativas se llevan a 0 al calcular las métricas |

### 2.2 Arquitectura

| Parámetro | Valor | Qué hace | Origen |
|---|---|---|---|
| `model` | `cudalstm` | LSTM de una capa, optimizado para GPU | Propuesta |
| `hidden_size` | 256 | Unidades de memoria del LSTM. Es el único hiperparámetro que se ajusta (2.6) | (2025) |
| `initial_forget_bias` | 3 | La compuerta de olvido arranca casi abierta (σ(3) ≈ 0,95), lo que ayuda a aprender dependencias largas como el flujo base | Ejemplos de NeuralHydrology |
| `output_dropout` | 0,4 | Apaga al azar el 40 % de las salidas del LSTM durante el entrenamiento, para reducir el sobreajuste | |
| `head`, `output_activation` | `regression`, `linear` | Capa final lineal: convierte los 256 estados en un valor de caudal | |

### 2.3 Entrenamiento

| Parámetro | Valor | Qué hace | Origen |
|---|---|---|---|
| `seq_length` | 365 | Días de clima que lee el modelo para predecir un día. Un año cubre las dos temporadas de lluvia y las dos secas; sin caudal rezagado, el modelo necesita esa historia para deducir cuánta agua guarda la cuenca | Chao et al. (2026); Wi et al.; Alzhanov et al. |
| `predict_last_n` | 1 | Solo el último día de cada ventana entra en la pérdida | |
| `batch_size` | 64 | Ventanas por actualización de pesos | Wi et al.; Alzhanov et al. |
| `epochs` | 30 | Máximo de pasadas por todas las ventanas | Propuesta |
| `optimizer`, `learning_rate` | Adam, 5 × 10⁻⁴ | Optimizador y tamaño del paso de cada actualización | Propuesta; Wi et al.; Alzhanov et al. |
| `loss` | `NSE` | NSE* de Kratzert: el error de cada día se divide por (σ_cuenca + 0,1)², para que las cuencas muy variables no dominen el entrenamiento | He et al. (2024); Yu & Tolson (2026) |
| `clip_gradient_norm` | 1 | Recorta los gradientes con norma mayor que 1, para evitar saltos bruscos de los pesos | Ejemplos de NeuralHydrology |
| `early_stopping` | `True`, paciencia 5, mínimo 5 épocas | Se detiene tras 5 épocas seguidas sin mejora en validación, nunca antes de la época 5 | Propuesta; paciencia y mínimo: decisión nuestra |
| `seed` | 42 | Misma inicialización y mismo orden de lotes en cada corrida | Reproducibilidad |

### 2.4 Validación, registro y hardware

- Se valida al final de cada época (`validate_every: 1`, lo exige la parada temprana) en las 248 cuencas.
- Métricas: NSE, KGE, RMSE, FHV, FLV y FMS; las tres últimas miden el sesgo en caudales altos, bajos y medios (Yilmaz et al., 2008). MAE no está en NeuralHydrology y se calculará aparte.
- Se guarda un checkpoint por época y el registro de TensorBoard en `runs/`, que está excluida de git.
- Entrena en la GPU (`cuda:0`) con `num_workers: 0`, porque en Windows los valores mayores pueden bloquear el notebook.

### 2.5 Configuración en código

In [6]:
def fechas(periodo):
    return [pd.Timestamp(f).strftime("%d/%m/%Y") for f in PERIODOS[periodo]]

config_base = {
    # ---- datos y períodos (2.1)
    "experiment_name": "camels_col_lstm",
    "run_dir": "runs",
    "dataset": "generic",
    "data_dir": "data/processed",
    "train_basin_file": "configs/basins/todas.txt",
    "validation_basin_file": "configs/basins/todas.txt",
    "test_basin_file": "configs/basins/todas.txt",
    "train_start_date": fechas("train")[0],
    "train_end_date": fechas("train")[1],
    "validation_start_date": fechas("validation")[0],
    "validation_end_date": fechas("validation")[1],
    "test_start_date": fechas("test")[0],
    "test_end_date": fechas("test")[1],
    "dynamic_inputs": FORZANTES,
    "target_variables": [OBJETIVO],
    "static_attributes": ATRIBUTOS,
    "clip_targets_to_zero": [OBJETIVO],
    # ---- arquitectura (2.2)
    "model": "cudalstm",
    "hidden_size": 256,
    "initial_forget_bias": 3,
    "output_dropout": 0.4,
    "head": "regression",
    "output_activation": "linear",
    # ---- entrenamiento (2.3)
    "seq_length": 365,
    "predict_last_n": 1,
    "batch_size": 64,
    "epochs": 30,
    "optimizer": "Adam",
    "learning_rate": 5.0e-4,
    "loss": "NSE",
    "clip_gradient_norm": 1,
    "early_stopping": True,
    "patience_early_stopping": 5,
    "minimum_epochs_before_early_stopping": 5,
    "seed": 42,
    # ---- validación, registro y hardware (2.4)
    "validate_every": 1,
    "validate_n_random_basins": len(cuencas),
    "metrics": ["NSE", "KGE", "RMSE", "FHV", "FLV", "FMS"],
    "save_weights_every": 1,
    "save_validation_results": False,
    "log_interval": 10,
    "log_tensorboard": True,
    "log_n_figures": 0,
    "device": "cuda:0",
    "num_workers": 0,
    "cache_validation_data": True,
}

def escribir_config(config, ruta):
    encabezado = "# Generado por proyecto-final-modelado.ipynb. Para cambiar un valor, edítalo en el notebook y vuelve a ejecutarlo.\n"
    ruta.write_text(encabezado + yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

escribir_config(config_base, ROOT / "configs" / "base.yml")
print((ROOT / "configs" / "base.yml").read_text(encoding="utf-8")[:1500], "...")

# Generado por proyecto-final-modelado.ipynb. Para cambiar un valor, edítalo en el notebook y vuelve a ejecutarlo.
experiment_name: camels_col_lstm
run_dir: runs
dataset: generic
data_dir: data/processed
train_basin_file: configs/basins/todas.txt
validation_basin_file: configs/basins/todas.txt
test_basin_file: configs/basins/todas.txt
train_start_date: 01/01/1981
train_end_date: 31/12/2010
validation_start_date: 01/01/2011
validation_end_date: 31/12/2015
test_start_date: 01/01/2016
test_end_date: 31/12/2022
dynamic_inputs:
- pr
- poten_evapo
- t_min
- t_max
target_variables:
- streamflow
static_attributes:
- log_area
- gravelius_index
- factor_form
- maximum_ele
- mean_ele
- minimum_ele
- streng_chanel
- equi_slope
- cn_catchment
- tc_Johnstone
- p_mean
- pet_mean
- aridity
- high_prec_freq
- high_prec_dur
- low_prec_freq
- low_prec_dur
- forest_perc
- nat_non_forest_form_perc
- agricul_livestock_perc
- non_vegeted_perc
- water_bodies_perc_cob
- unconso_depo_perc
- hypabyssal_rock_perc

### 2.6 Ajuste de `hidden_size` (pendiente)

Según la propuesta, `hidden_size` se elige con el período de validación de las cuencas que **nunca** se retienen en la validación cruzada espacial (los dos sistemas de ríos más grandes), porque la complejidad del LSTM debe corresponder al número de cuencas de entrenamiento (He et al., 2024).

Antes hay que identificar el anidamiento (qué cuencas están conectadas) con los polígonos de `03_CAMELS_COL_Basin_boundary`. La propuesta hablaba de 216 cuencas fijas sobre 346; con las 248 cuencas filtradas, ese número cambia y hay que recalcularlo.

## 3. Prueba de humo

Antes del entrenamiento completo (varias horas) se corre uno corto para verificar toda la cadena: lectura de datos, entrenamiento, validación y checkpoints. La configuración es la misma de la sección 2 con estos cambios:

| Parámetro | Modelo principal | Prueba de humo | Por qué |
|---|---|---|---|
| `experiment_name` | `camels_col_lstm` | `camels_col_humo` | Separar las corridas |
| `train/validation/test_basin_file` | `todas.txt` (248) | `humo_20.txt` (20) | 4 cuencas por región |
| `epochs` | 30 | 2 | Solo verificar que la pérdida baja |
| `validate_n_random_basins` | 248 | 20 | |
| `early_stopping` | `True` | `False` | Con 2 épocas no aplica |
| `log_n_figures` | 0 | 3 | Ver algunos hidrogramas |

In [7]:
config_humo = {
    **config_base,
    "experiment_name": "camels_col_humo",
    "train_basin_file": "configs/basins/humo_20.txt",
    "validation_basin_file": "configs/basins/humo_20.txt",
    "test_basin_file": "configs/basins/humo_20.txt",
    "epochs": 2,
    "validate_n_random_basins": 20,
    "early_stopping": False,
    "log_n_figures": 3,
}
escribir_config(config_humo, ROOT / "configs" / "humo.yml")
print("Escrito configs/humo.yml")

Escrito configs/humo.yml


In [ ]:
EJECUTAR_PRUEBA_DE_HUMO = True

if EJECUTAR_PRUEBA_DE_HUMO:
    import torch
    from neuralhydrology.nh_run import start_run

    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")
    start_run(config_file=ROOT / "configs" / "humo.yml")
else:
    print("Prueba de humo desactivada: instala PyTorch y NeuralHydrology y cambia la bandera a True.")

GPU: NVIDIA GeForce RTX 2060 with Max-Q Design
2026-11-10 16:56:55,886: Logging to runs\camels_col_humo_1011_165655\output.log initialized.
2026-11-10 16:56:55,888: ### Folder structure created at runs\camels_col_humo_1011_165655
2026-11-10 16:56:55,889: ### Run configurations for camels_col_humo
2026-11-10 16:56:55,892: experiment_name: camels_col_humo
2026-11-10 16:56:55,894: run_dir: runs\camels_col_humo_1011_165655


2026-11-10 16:56:55,896: dataset: generic
2026-11-10 16:56:55,898: data_dir: data\processed
2026-11-10 16:56:55,900: train_basin_file: configs\basins\humo_20.txt
2026-11-10 16:56:55,903: validation_basin_file: configs\basins\humo_20.txt
2026-11-10 16:56:55,905: test_basin_file: configs\basins\humo_20.txt
2026-11-10 16:56:55,907: train_start_date: 1981-01-01 00:00:00
2026-11-10 16:56:55,911: train_end_date: 2010-12-31 00:00:00
2026-11-10 16:56:55,912: validation_start_date: 2011-01-01 00:00:00
2026-11-10 16:56:55,914: validation_end_date: 2015-12-31 00:00:00
2026-11-10 16:56:55,917: test_start_date: 2016-01-01 00:00:00
2026-11-10 16:56:55,921: test_end_date: 2022-12-31 00:00:00
2026-11-10 16:56:55,922: dynamic_inputs: ['pr', 'poten_evapo', 't_min', 't_max']
2026-11-10 16:56:55,924: target_variables: ['streamflow']
2026-11-10 16:56:55,926: static_attributes: ['log_area', 'gravelius_index', 'factor_form', 'maximum_ele', 'mean_ele', 'minimum_ele', 'streng_chanel', 'equi_slope', 'cn_catchme

**Qué revisar después de la prueba de humo:**

1. La pérdida de entrenamiento baja entre la época 1 y la 2.
2. Se reportan métricas de validación (NSE, KGE…) sin errores.
3. Existe `runs/camels_col_humo_<fecha>/` con `model_epoch001.pt` y `model_epoch002.pt`.
4. **Tiempo por época:** escalarlo de 20 a 248 cuencas para estimar el entrenamiento completo.
5. **Tamaño de lote:** repetir con `batch_size: 256` y comparar el tiempo. Si es mucho más rápido, el equipo decide si conviene cambiarlo (verificando que el desempeño en validación no empeore).

**Resultado** (10/11/2026, `runs/camels_col_humo_1011_165655`): la pérdida de entrenamiento bajó de 0,052 a 0,041; la validación reportó NSE 0,46, KGE 0,48 y RMSE 1,79 mm/día (medianas de las 20 cuencas, época 2), y se guardaron los dos checkpoints. Cada época tomó unos 2,5 minutos. Las 248 cuencas tienen 14 veces más días de entrenamiento, así que con `batch_size: 64` se esperan unos 35 minutos por época. Falta el punto 5.

## 4. Próximos pasos

1. Definir la estrategia de modelado.
2. Probar `batch_size: 256` en la prueba de humo.
3. Identificar el anidamiento (sistemas de ríos) con los polígonos de cuenca y recalcular las particiones espaciales con 248 cuencas.
4. Baseline: climatología diaria por cuenca.
5. Ajuste de `hidden_size`.
6. Entrenamiento del modelo principal y evaluación en 2016 a 2022.
7. Validación cruzada espacial (5 particiones).
8. MAE y análisis de resultados por región.